# Synchroniser le Dataset Lab vers Hugging Face

Ce notebook prépare une synchronisation append-only vers le dépôt privé ethanzxv/pixelart-dataset. Il peut publier les snapshots d’annotations et les deux taxonomies, ainsi que des fichiers du dataset local choisis explicitement. Il ne supprime aucun fichier distant et ne pousse jamais automatiquement toutes les images.

Le mode initial est une prévisualisation. Vérifie les fichiers, les tailles et le nombre de lignes avant de passer DRY_RUN à False. Les annotations en cours sont lues comme un snapshot cohérent; une dernière ligne incomplète pendant une écriture est ignorée et sera incluse au prochain sync.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import subprocess

from dotenv import load_dotenv
from huggingface_hub import CommitOperationAdd, HfApi

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in (cwd, *cwd.parents) if (p / "notebook/dataset/taxonomy/taxonomies.json").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Racine iris-pixelart introuvable depuis le dossier courant.")

DATASET_NOTEBOOK = PROJECT_ROOT / "notebook/dataset"
TAXONOMY_DIR = DATASET_NOTEBOOK / "taxonomy"
WORK_DIR = TAXONOMY_DIR / "taxonomy_work"
DATASET_ROOT = PROJECT_ROOT / "dataset-viewer/datasets/pixelart-dataset/.gitrepo"
REPO_ID = "ethanzxv/pixelart-dataset"
REPO_TYPE = "dataset"

# La prévisualisation ne modifie ni le dépôt HF ni les fichiers sources.
DRY_RUN = True

# Laisser vide pour ne pas synchroniser de nouveaux assets bruts.
# Ajouter seulement des chemins relatifs à la racine du dépôt local, par ex.
# DATA_FILES_TO_SYNC = ["data/mon-pack/sprite.png", "data/mon-pack/sprite.txt"]
DATA_FILES_TO_SYNC = []

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
STAGING_DIR = WORK_DIR / "dataset_lab_sync" / RUN_ID
ANNOTATION_SOURCES = [
    (WORK_DIR / "annotations_v3_3_strata_gyro_s.jsonl", "annotations_v3_3_partial.jsonl"),
    (WORK_DIR / "annotations_v3_4_full_color.jsonl", "annotations_v3_4_color_partial.jsonl"),
]
TAXONOMY_SOURCES = [
    (TAXONOMY_DIR / "taxonomies.json", "taxonomies.json"),
    (TAXONOMY_DIR / "taxonomies_couleurs.json", "taxonomies_couleurs.json"),
]

# Charger les secrets depuis l’environnement ou le fichier local ignoré par Git.
for env_file in (
    DATASET_NOTEBOOK / ".env",
    DATASET_NOTEBOOK.parent / ".env",
    PROJECT_ROOT / "notebook/.env",
    Path.home() / "iris-sprite-lora/training/.env",
):
    if env_file.is_file():
        load_dotenv(env_file, override=False)
HF_TOKEN = (
    os.environ.get("HF_TOKEN")
    or os.environ.get("HF_ACCESS_TOKEN")
    or os.environ.get("HUGGINGFACE_HUB_TOKEN")
)
print(f"Racine dataset: {DATASET_ROOT}")
print(f"Mode: {'prévisualisation' if DRY_RUN else 'publication'} · run {RUN_ID}")

## Contrôler les assets sélectionnés

Les assets bruts sont optionnels et doivent être listés explicitement. Le dépôt local contient des fichiers Git LFS; le code refuse de téléverser un pointeur LFS à la place d’une image réelle. Les chemins sont conservés à l’identique dans le dépôt HF.

In [ ]:
def resolve_dataset_file(relative_path):
    candidate = (DATASET_ROOT / relative_path).resolve()
    try:
        rel = candidate.relative_to(DATASET_ROOT.resolve()).as_posix()
    except ValueError as exc:
        raise ValueError(f"Chemin hors dataset refusé: {relative_path}") from exc
    if not rel.startswith(("data/", "source_sheets/")):
        raise ValueError(f"Seuls data/ et source_sheets/ sont autorisés: {rel}")
    if not candidate.is_file():
        raise FileNotFoundError(candidate)
    with candidate.open("rb") as stream:
        if stream.read(42).startswith(b"version https://git-lfs.github.com/spec/v1"):
            raise RuntimeError(f"Asset LFS non téléchargé: {rel}")
    return rel, candidate

selected_assets = [resolve_dataset_file(path) for path in DATA_FILES_TO_SYNC]
print(f"{len(selected_assets)} asset(s) brut(s) sélectionné(s).")
for rel, path in selected_assets[:30]:
    print(f"- {rel} · {path.stat().st_size:,} octets")
if len(selected_assets) > 30:
    print(f"... et {len(selected_assets) - 30} autres")

## Préparer un snapshot cohérent

Chaque ligne JSONL valide est copiée dans un nouveau dossier horodaté. Une ligne finale incomplète est écartée, ce qui permet de synchroniser pendant qu’un run ajoute des résultats. Les taxonomies et un manifeste rendent le snapshot autonome et traçable.

In [ ]:
def snapshot_jsonl(source, destination):
    rows = 0
    destination.parent.mkdir(parents=True, exist_ok=True)
    snapshot_size = source.stat().st_size
    with source.open("rb") as src, destination.open("w", encoding="utf-8") as dst:
        while src.tell() < snapshot_size:
            raw = src.readline(snapshot_size - src.tell())
            # Une ligne sans fin de ligne peut être en cours d’écriture.
            if not raw.endswith(b"\n"):
                break
            try:
                row = json.loads(raw)
            except (json.JSONDecodeError, UnicodeDecodeError) as exc:
                raise ValueError(f"Ligne JSON invalide dans le snapshot de {source}") from exc
            if not isinstance(row, dict):
                raise ValueError(f"Une ligne n'est pas un objet JSON dans {source}")
            dst.write(json.dumps(row, ensure_ascii=False, separators=(",", ":")) + "\n")
            rows += 1
    return rows
STAGING_DIR.mkdir(parents=True, exist_ok=False)
snapshot_counts = {}
upload_items = []
for source, remote_name in ANNOTATION_SOURCES:
    if not source.is_file():
        continue
    dest = STAGING_DIR / remote_name
    count = snapshot_jsonl(source, dest)
    if count:
        snapshot_counts[remote_name] = count
        upload_items.append((dest, remote_name))

for source, remote_name in TAXONOMY_SOURCES:
    if not source.is_file():
        raise FileNotFoundError(source)
    dest = STAGING_DIR / remote_name
    dest.write_bytes(source.read_bytes())
    upload_items.append((dest, remote_name))

status_path = WORK_DIR / "run_full_color_status.json"
run_status = json.loads(status_path.read_text(encoding="utf-8")) if status_path.is_file() else None
manifest = {
    "synced_at_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_repo": REPO_ID,
    "run_id": RUN_ID,
    "annotations": snapshot_counts,
    "strata_run_status": run_status,
    "taxonomies": {name: hashlib.sha256((TAXONOMY_DIR / name).read_bytes()).hexdigest() for name in ("taxonomies.json", "taxonomies_couleurs.json")},
    "raw_assets": [rel for rel, _ in selected_assets],
    "note": "Append-only snapshot. No remote files are deleted."
}
manifest_path = STAGING_DIR / "run_manifest.json"
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
upload_items.append((manifest_path, "run_manifest.json"))

for rel, source in selected_assets:
    upload_items.append((source, rel))

print("Annotations copiées:", snapshot_counts)
print("Fichiers prêts:", len(upload_items))
print("Taille totale:", f"{sum(path.stat().st_size for path, _ in upload_items):,} octets")
print("Dossier HF:", f"annotations/dataset_lab_sync/{RUN_ID}")

## Publier

En mode DRY_RUN, cette cellule affiche seulement la destination et les fichiers. Pour publier, passe DRY_RUN à False puis réexécute les cellules de configuration jusqu’ici avant d’exécuter cette dernière cellule. Le commit crée un dossier horodaté et ajoute les assets sélectionnés à leurs chemins; aucun fichier existant n’est supprimé.

In [ ]:
if DRY_RUN:
    print("Prévisualisation seulement. Aucun fichier n’a été envoyé.")
    for path, remote_name in upload_items:
        print(f"- {remote_name} · {path.stat().st_size:,} octets")
else:
    if not HF_TOKEN:
        raise RuntimeError("Jeton HF absent. Configure HF_TOKEN, HF_ACCESS_TOKEN ou hf auth login.")
    api = HfApi(token=HF_TOKEN)
    revision = api.repo_info(REPO_ID, repo_type=REPO_TYPE).sha
    remote_dir = f"annotations/dataset_lab_sync/{RUN_ID}"
    operations = []
    for local_path, remote_name in upload_items:
        if remote_name.startswith(("data/", "source_sheets/")):
            remote_path = remote_name
        else:
            remote_path = f"{remote_dir}/{remote_name}"
        operations.append(CommitOperationAdd(path_in_repo=remote_path, path_or_fileobj=str(local_path)))
    commit = api.create_commit(
        repo_id=REPO_ID,
        repo_type=REPO_TYPE,
        parent_commit=revision,
        operations=operations,
        commit_message=f"Sync Dataset Lab snapshot ({sum(snapshot_counts.values())} annotations)",
        commit_description=f"Append-only dataset snapshot {RUN_ID}; includes selected taxonomy annotations and taxonomy files.",
    )
    print(f"Synchronisation terminée: {commit.commit_url}")

### Interpréter la progression Strata

Le run local de taxonomie peut s’interrompre si la machine locale ou Strata s’arrête. Les annotations JSONL déjà écrites restent récupérables. Le runner reprend à la première image absente du fichier et synchronise une nouvelle copie horodatée à la fin du lot. Pour une reprise, démarrer Strata sur la machine distante puis lancer le script indiqué dans le statut du run.